# 1. Quick start: three cities in five lines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hpa-code/simcitynet/blob/main/docs/tutorials/01_quickstart.ipynb)

*Run it in Colab (free GPU) to see every figure; the website shows the saved outputs.*

This notebook embeds three street photos with SimCityNet, shows what the model sees in each
one, and compares them. It takes a few minutes on a Colab GPU, most of it downloading the
models the first time.

In [ ]:
# Installs simcitynet with all models (PyTorch, Transformers, Ultralytics YOLO).
# In Colab, choose Runtime > Change runtime type > GPU for speed.
# Uses PyPI; until the first PyPI release it installs from GitHub instead.
!pip install -q "simcitynet[all]" || pip install -q "simcitynet[all] @ git+https://github.com/hpa-code/simcitynet.git"

## Get three example photos

`fetch_examples` downloads openly licensed street scenes from Wikimedia Commons and records
their authors and licences in `CREDITS.csv`. Use your own photos instead by passing their
paths to `embed` below.

In [ ]:
import pandas as pd
from simcitynet.datasets import fetch_examples, credits_table

paths = fetch_examples("examples", names=["amsterdam", "zurich", "dubai"])
pd.DataFrame(credits_table("examples"))[["name", "author", "license", "source_url"]]

## Embed

`SimCityNet()` loads its models lazily, the first time they are needed.
`embed` accepts paths, folders, glob patterns, PIL images or NumPy arrays.

In [ ]:
from simcitynet import SimCityNet

scn = SimCityNet()          # device="auto": GPU if available
emb = scn.embed(paths)
emb

Every image becomes 33 numbers. Each column has a meaning (see *The 33 dimensions* in the
docs). Here are a few:

In [ ]:
emb.raw[["vegetation", "person", "car", "bus", "road_share", "sidewalk_share", "building_share"]].round(3)

`summary()` condenses the vector into eight readable quantities:

In [ ]:
emb.summary().round(2)

## What SimCityNet sees

A scene card shows the photo, the vegetation it found (brighter green = closer to the camera),
the road / sidewalk / building segmentation, the people and vehicles it counted, the depth
weights, and the five dominant colours.

In [ ]:
emb.plot.scene_card("zurich")

In [ ]:
emb.plot.scene_card("dubai")

## Compare

Cosine similarity on the normalised vectors (the paper's default):

In [ ]:
emb.similarity().round(3)

In [ ]:
emb.plot.heatmap()

The colour palettes can also be compared directly with the Earth Mover's distance, in RGB
units (0 = identical palettes):

In [ ]:
emb.distance("color_emd").round(1)

In [ ]:
emb.plot.radar()

## Save

The CSV holds the 33 features; the JSON next to it records the package, model and library
versions so the result can be reproduced.

In [ ]:
emb.save("results/three_cities")
!head -c 600 results/three_cities.json

Next: **[2. Europe vs the Middle East](02_europe_vs_middle_east.ipynb)** compares six cities
from the two regions studied in the paper.